<a href="https://colab.research.google.com/github/singhshahrahul/AI-ML/blob/main/09_transfer_learning_feature_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tools

In [ ]:
import tensorflow as tf
import keras
import tensorflow_hub as hub
from tensorflow.keras import layers
import numpy as np
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess

print("TF: ", tf.__version__)
print("TF Hub: ", hub.__version__)
print("Numpy: ", np.__version__)

TF:  2.20.0
TF Hub:  0.16.1
Numpy:  2.1.3


#Transfer Learning with TensrFlow

Part 1. Feature Extraction

In [ ]:
import zipfile

#Get data (10%)
!wget  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

#Unzip
zip_ref = zipfile.ZipFile("10_food_classes_10_percent.zip", "r")
zip_ref.extractall()

--2026-08-23 14:25:42--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.99.207, 173.194.202.207, 173.194.203.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.99.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip’

10_food_classes_10_ 100%[===================>] 160.74M   168MB/s    in 1.0s    

2026-08-23 14:25:43 (168 MB/s) - ‘10_food_classes_10_percent.zip’ saved [168546183/168546183]



In [ ]:
zip_ref.close()

In [ ]:
import os
for dirpath, dirnames, filenames in os.walk("10_food_classes_10_percent"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in {dirpath}")

There are 2 directories and 0 images in 10_food_classes_10_percent
There are 10 directories and 0 images in 10_food_classes_10_percent/test
There are 0 directories and 250 images in 10_food_classes_10_percent/test/steak
There are 0 directories and 250 images in 10_food_classes_10_percent/test/pizza
There are 0 directories and 250 images in 10_food_classes_10_percent/test/sushi
There are 0 directories and 250 images in 10_food_classes_10_percent/test/chicken_curry
There are 0 directories and 250 images in 10_food_classes_10_percent/test/chicken_wings
There are 0 directories and 250 images in 10_food_classes_10_percent/test/grilled_salmon
There are 0 directories and 250 images in 10_food_classes_10_percent/test/ramen
There are 0 directories and 250 images in 10_food_classes_10_percent/test/ice_cream
There are 0 directories and 250 images in 10_food_classes_10_percent/test/fried_rice
There are 0 directories and 250 images in 10_food_classes_10_percent/test/hamburger
There are 10 directori

Preparing the data



In [ ]:
img_shape = (224, 224)
batch_size = 32

train_dir = "10_food_classes_10_percent/train/"
test_dir = "10_food_classes_10_percent/test"

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    label_mode="categorical",
    image_size=img_shape,
    batch_size=batch_size
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    label_mode="categorical",
    image_size=img_shape,
    batch_size=batch_size
)

num_classes = len(train_data.class_names)

train_data = train_data.prefetch(tf.data.AUTOTUNE)
test_data = test_data.prefetch(tf.data.AUTOTUNE)

Found 750 files belonging to 10 classes.
Found 2500 files belonging to 10 classes.


In [ ]:
#Resnet 50 V2 featuree vector
resnet_url = "https://tfhub.dev/google/imagenet/resnet_v2_50/feature_vector/4"

#EfficientNetB0 feature vector (version 1)
efficientnet_url = "https://tfhub.dev/tensorflow/efficientnet/b0/feature-vector/1"

In [ ]:
#Function for creating model by taking url
def create_model(model_url, num_classes=10):
  input_shape = img_shape + (3,) #(224, 224, 3)

  feature_extractor = hub.KerasLayer(resnet_url, trainable=False)

  inputs = tf.keras.Input(shape=input_shape, name = "input_image")
  x = tf.keras.layers.Rescaling(1./255)(inputs)
  x = tf.keras.layers.Lambda(lambda img: feature_extractor(img))(x)
  outputs = tf.keras.layers.Dense(num_classes, activation = "softmax", name="output")(x)
  model = tf.keras.Model(inputs=inputs, outputs=outputs)

  return model

In [ ]:
#create model
resnet_model = create_model(resnet_url, num_classes = num_classes)

#compile
resnet_model.compile(loss="categorical_crossentropy",
                     optimizer = tf.keras.optimizers.Adam(),
                     metrics=["accuracy"])

In [ ]:
#fit the model
resnet_history = resnet_model.fit(train_data, epochs=5,
                                  validation_data=test_data)

Epoch 1/5
17/24 ━━━━━━━━━━━━━━━━━━━━ 8:28 73s/step - accuracy: 0.2195 - loss: 2.2885

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(history):
  loss = history.history["loss"]
  val_loss = history.history["val_loss"]

  accuracy = history.history['accuracy']
  val_accuracy = history.history["val_accuracy"]

  epochs = range(len(history.history['loss']))

  plt.plot(epochs, loss, label='training_loss')
  plt.plot(epochs, val_loss, label='val_loss')
  plt.title('Loss')
  plt.xlabel("Epochs")
  plt.legend()

  plt.figure()
  plt.plot(epochs, accuracy, label='training_accuracy')
  plt.plot(epochs, val_accuracy, label="val_accuracy")
  plt.title("Accuracy")
  plt.xlabel("Epochs")
  plt.legend()

In [ ]:
plot_loss_curves(resnet_history)

In [ ]:
#EfficientNetB0 feature vector (version 2)
efficientnet_url = "https://tfhub.dev/google/imagenet/efficientnet_v2_imagenet1k_b0/feature_vector/2"

In [ ]:
#create model
efficientnet_model = create_model(model_url=efficientnet_url, num_classes=num_classes)

#compile
efficientnet_model.compile(loss="categorical_crossentropy",
                     optimizer = tf.keras.optimizers.Adam(),
                     metrics=["accuracy"])

In [ ]:
#fit
efficientnet_history = efficientnet_model.fit(train_data,
                                              epochs = 5,
                                              validation_data = test_data)

In [ ]:
plot_loss_curves(efficientnet_history)

In [ ]:
resnet_model.summary()

In [ ]:
efficientnet_model.summary()